# Workshop: Gemma from Scratch
## Notebook 1: The Math of Attention (Weights & Softmax)

**Estimated Time: 15 minutes**

In this notebook, we will implement the core of the Transformer: **Scaled Dot-Product Attention**. This is the mechanism that allows the model to relate different positions of a sequence to compute a representation of the sequence.

### Learning Objectives:
1. Understand the Query (Q), Key (K), and Value (V) analogy.
2. Implement the attention score calculation.
3. Understand why scaling by $\sqrt{d_k}$ is necessary.
4. **NEW: Learn about Logit Soft-Capping for training stability.**
5. Visualize how Softmax creates an attention map.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

# Set seed for reproducibility
_ = torch.manual_seed(42)

### 1. The Q, K, V Analogy

Imagine you are in a library:
- **Query (Q)**: The topic you are searching for (e.g., "How do transformers work?").
- **Key (K)**: The labels on the spines of the books (e.g., "Deep Learning", "Gardening", "Attention Mechanisms").
- **Value (V)**: The actual content inside the books.

Attention matches your **Query** against all **Keys**, computes a similarity score, and uses that score to decide how much of each **Value** you should read.

In [ ]:
# Let's define some dummy dimensions
batch_size = 1
seq_len = 5
head_dim = 16

# Randomly initialize Q, K, V
Q = torch.randn(batch_size, seq_len, head_dim)
K = torch.randn(batch_size, seq_len, head_dim)
V = torch.randn(batch_size, seq_len, head_dim)

print(f"Shape of Q: {Q.shape}")

### 2. Computing Attention Scores

The similarity is computed using the dot product: $Score = Q K^T$

We need to transpose K so that we multiply each Query vector by each Key vector.

In [ ]:
# Compute scores
# (B, T, D) @ (B, D, T) -> (B, T, T)
scores = torch.matmul(Q, K.transpose(-2, -1))

print(f"Scores shape: {scores.shape}")
print("Sample scores:\n", scores[0, :2, :2])

### 3. Scaling & Soft-Capping

As $d_k$ (head_dim) gets larger, the magnitude of the dot products grows. This pushes the softmax function into regions where it has extremely small gradients. 

To counteract this, we do two things:
1. **Scale** by $1/\sqrt{d_k}$.
2. **Soft-Cap** (used in Gemma 2/3): We constrain the logits using a `tanh` function to prevent them from exploding during training.

In [ ]:
scaled_scores = scores / math.sqrt(head_dim)

# Logit Soft-Capping (Gemma 2 style)
def soft_cap(logits, cap=50.0):
    return cap * torch.tanh(logits / cap)

capped_scores = soft_cap(scaled_scores)
print("Capped scores (top 2x2):\n", capped_scores[0, :2, :2])

### 4. Softmax and Attention Weights

Softmax turns scores into probabilities that sum to 1. These are our **Attention Weights**.

In [ ]:
attention_weights = F.softmax(capped_scores, dim=-1)

print("Attention Weights (Sum of row 1):", attention_weights[0, 0].sum().item())

plt.imshow(attention_weights[0].detach().numpy())
plt.colorbar()
plt.title("Attention Map")
plt.xlabel("Keys")
plt.ylabel("Queries")
plt.show()

### 5. The Output

Finally, we multiply the weights by the Values to get the context-aware representation.
$Output = Weights \times V$

In [ ]:
output = torch.matmul(attention_weights, V)
print(f"Output shape: {output.shape}")

### Exercise:
Write a single function `scaled_dot_product_attention(Q, K, V, soft_cap_val=50.0)` that performs all these steps.

**Hints:**
1. Use `torch.matmul` for matrix multiplication.
2. Scale by `math.sqrt(Q.shape[-1])`.
3. Apply the `soft_cap` formula: `cap * tanh(logits / cap)`.
4. Apply `F.softmax` on the last dimension.

In [ ]:
def scaled_dot_product_attention(Q, K, V, soft_cap_val=50.0):
    # Your code here
    pass

# Test your function
try:
    test_output = scaled_dot_product_attention(Q, K, V)
    if test_output is not None:
        assert torch.allclose(test_output, output, atol=1e-6)
        print("✅ Success! Your implementation matches the results.")
    else:
        print("Implement the function to test it.")
except Exception as e:
    print(f"❌ Error: {e}")

<details>
<summary><b>Click to see solution</b></summary>

```python
def scaled_dot_product_attention(Q, K, V, soft_cap_val=50.0):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if soft_cap_val is not None:
        scores = soft_cap_val * torch.tanh(scores / soft_cap_val)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V)
```
</details>